<table>
  <tr>
    <td><div align="left"><font size="30">Point clouds</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200" height="158" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2026

Robotics, Vision & Control: Python, see Chapter 14.7

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`). If you're running locally you need to have these packages installed.

<div class="alert alert-info">
If you're running on CoLab or in the browser we have to first install machinevisiontoolbox, which is not preinstalled, so the next cell will be slow.
</div>

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'ipympl', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt
plt.close("all")   # re-running the notebook from the top starts with no figures

import numpy as np
from machinevisiontoolbox import PointCloud

# display result of assignments
if COLAB:
    %config ZMQInteractiveShell.ast_node_interactivity = 'last_expr_or_assign'
# make NumPy display a bit nicer
np.set_printoptions(linewidth=100, formatter={'float': lambda x: f"{x:10.4g}" if abs(x) > 1e-10 else f"{0:10.4g}"})

# Introduction

This notebook introduces the toolbox's `PointCloud` class which is a wrapper around the [Open3D package](https://www.open3d.org/docs/release/). Open3D is an open-source library that supports rapid development of software that deals with 3D data. The Open3D frontend exposes a set of carefully selected data structures and algorithms in both C++ and Python. The backend is highly optimized and is set up for parallelization.

The `PointCloud` class is analogous to the `Image` class which is a wrapper around `OpenCV`.  However, since Open3D is already object oriented this task is quite straightforward compared to wrapping `OpenCV` Python binding which is function style.

The `PointCloud` class provides a subset of commonly used methods but does not attempt to wrap
every Open3D method.  Instead, any method not known to the `PointCloud` class such as `pc.some_method()` will fall through to the native Open3D method `some_method()` on the encapsulated Open3D object, and pass through all parameters.  If the return value is an Open3D point cloud, that is wrapped and returned as a `PointCloud` instance.

Open3D can import and export NumPy arrays, and points are rows.  This is different to the MVTB convention
of points as columns.  The MVTB wrappers handle this transposition, but those methods that "fall through" to Open3D will use the Open3D convention.


# Loading and displaying a point cloud

We will load a PLY file (ASCII format readable text) containing points and triangular faces.  PLY is a common
file format, there are many models available online, and many excellent desktop viewing tools.

It's a convention in point cloud processing code to append `_pcd` to a variable that contains a point cloud.  The point cloud represents
the famous "Stanford bunny" which was digitized in 1994 at Stanford and is well known in the computer graphics community.


In [ ]:
bunny_pcd = PointCloud.Read("bunny.ply")
print(bunny_pcd)

`bunny_pcd` is a Machine Vision Toolbox `PointCloud` object which wraps an Open3D point cloud object that we see
has over 35 thousand points.

Just as we would for an image, we use the `disp` method to display it.

In [ ]:
bunny_pcd.disp(notebook=True)

Under the hood, this is plotted using Plotly.  Using the mouse you can rotate and zoom the point cloud.

# Point cloud processing

## Computing a bounding box

We can fit an axis aligned bounding box to the point cloud

In [ ]:
bbox = bunny_pcd.get_axis_aligned_bounding_box()
print(bbox)

In [ ]:
bbox.max_bound - bbox.min_bound

The bunny is about 16x16x12 cm in size

In [ ]:
bunny_pcd.voxel_grid(voxel_size=0.01).disp(notebook=True, mesh_show_wireframe=True)

## Subsampling

Sometimes point clouds have more points than we want, more points increases storage and can slow
subsequent processing operations.

One approach is voxel downsampling. It reduces the number of points in a cloud while keeping its overall 3D shape and creating a uniform point density. It runs in two steps:
1. Voxel Grid Bucketing: It divides 3D space into a regular grid of tiny cubes (voxels), where each cube has side length `voxel_size` (in the same units as the point coordinates).
2. Point Averaging: For every occupied cube, it combines all points inside that cube into one single output point by averaging their [x, y, z] coordinates.

We will downsample with 1cm voxels

In [ ]:
sparse_pcd = bunny_pcd.downsample_voxel(voxel_size=0.01)
sparse_pcd.disp(notebook=True)
print(sparse_pcd)

Subsampling can also be achieved by random sampling, which keeps a randomly chosen fraction of the points, here 5%.

In [ ]:
sparse2_pcd = bunny_pcd.downsample_random(fraction=0.05)
sparse2_pcd.disp(notebook=True)
print(sparse2_pcd)

By default every random subsampling is different.  For repeatable random downsample you can set the seed

In [ ]:
sparse2_pcd = bunny_pcd.downsample_random(fraction=0.05, seed=0)

## Surface normals

Open3D computes point cloud normals in three steps using Principal Component Analysis (PCA):
1. Find Neighbors: For each point, Open3D uses a KD-Tree to gather nearby points within a specified radius or nearest-neighbor count.
2. Build Covariance Matrix: It calculates the centroid of those neighbors and constructs a 3 × 3 covariance matrix to measure how the points spread out in 3D space.
3. Extract Eigenvector: It decomposes the matrix and selects the eigenvector with the smallest eigenvalue as the surface normal, since it points in the direction of least variance perpendicular to the local plane.

In [ ]:
sparse_pcd.estimate_normals(radius=0.02, max_nn=30)
print(sparse_pcd.normals)

The normals are the columns of a 3xN matrix.  We can use a 3D quiver plot to to display the normals, the tail of each vector is a point-cloud point.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
p = sparse_pcd.points
n = sparse_pcd.normals
x, y, z = p[0,:], p[1, :], p[2, :]
u, v, w = n[0, :], n[1, :], n[2, :]
ax.quiver(
    x, y, z, u, v, w,
    length=0.005,             # Scales the arrow length down (default is 1.0)
    normalize=True,         # Forces all arrows to have the exact same length
    arrow_length_ratio=0.4  # Adjusts arrowhead size relative to the shaft (default is 0.3)
)
plt.show()

## Finding objects in a scene

This synthetic scene has a ground plane and three objects sitting on that plane: a box, a cylinder
and a sphere.  The point coordinates have been "noised" with a small amount of Gaussian noise.


In [ ]:
scene = PointCloud.Read("scene.ply")
scene.disp(notebook=True)
print(scene)

We can readily identify the dominant plane  using RANSAC and giving a tolerance for how variation is
allowed for points to deviate from the estimated plane.

In [ ]:
ground_plane, ground, objects = scene.segment_plane(distance_threshold=0.05)
print(ground_plane)
print(ground)


There are 3 return values:

1. parameters of the plane expressed as $(a,b,c,d), aX+bY+cZ+d=0$ and the estimated plane is close to $z=0$.
2. a new point cloud representing the ground plane
3. a new point cloud representing the remainder of the points

If we look at the ground plane point cloud

In [ ]:
ground.disp(notebook=True)

we see that some object points that were close to the ground plane have been included.

Now we look at the remainder point cloud

In [ ]:
print(objects)
objects.disp(notebook=True)

and we can see the three geometric shapes floating in space.

Next we cluster those points using DBSCAN clustering, and then fit a bounding box to each cluster.

DBSCAN needs two parameters.  `min_points` is the number of points needed within a distance `eps` of a point for it to count as being in a dense region, and a cluster is a chain of such points whose neighbourhoods overlap.  Points that are not reached are noise and get the label -1.

The value of `eps` matters.  It must be larger than the spacing of points within an object, or the objects break into fragments, and smaller than the gap between objects, or they merge.  We can choose it from the data with a *k-distance plot*.  For every point we find the distance to its $k$-th nearest neighbour, with $k$ equal to `min_points`, and sort them.  Most points are in dense regions and have a small distance, the few isolated points have a large one, and the knee in the curve is a good value for `eps`.

In [ ]:
from scipy.spatial import cKDTree

min_points = 10
pts = objects.points.T                     # one row per point, as SciPy expects
dist, _ = cKDTree(pts).query(pts, k=min_points)
kdist = np.sort(dist[:, -1])               # distance to the k-th nearest neighbour

plt.figure()
plt.plot(kdist)
plt.xlabel("points, sorted by distance")
plt.ylabel(f"distance to {min_points}th nearest neighbour (m)")
plt.grid(True)

The curve is flat at about 0.2 m, where the points are in dense regions, and then climbs steeply at about 0.3 m to the isolated points.  We choose `eps` just above the knee, which is where we got 0.3 from.

In [ ]:
labels = np.array(
    objects.cluster_dbscan(eps=0.3, min_points=10, print_progress=False)
)

and the result is a 1d-array that contains a label for the corresponding point cloud point

In [ ]:
print(labels.shape)
print(np.unique(labels, return_counts=True))

We see that it has values 0, 1, and 2 which correspond to the three shapes.

Another option is to perform a sweep over `eps` and see how clusters are found.

In [ ]:
for eps in np.arange(0.1, 1.0, 0.05):
    trial = np.array(
        objects.cluster_dbscan(eps=eps, min_points=10, print_progress=False)
    )
    print(f"eps={eps:.2f}: Detected {trial.max() + 1} object clusters")

The loop could exit when the number of clusters matches the known number of objects in the scene.

## Coloring the clusters

We give the ground, each cluster and the noise points their own colors, and join them into one point cloud for display.

In [ ]:
cmap = plt.get_cmap("tab10")
parts = [ground.paint([0.6, 0.6, 0.6])]             # the ground is gray
for k in range(labels.max() + 1):                   # each cluster gets its own color
    parts.append(PointCloud(objects.points[:, labels == k]).paint(cmap(k)[:3]))
parts.append(PointCloud(objects.points[:, labels < 0]).paint([0, 0, 0]))  # noise is black

# the + operator joins point clouds, and keeps the color of each point
colored = sum(parts)
colored.disp(notebook=True)

In [ ]:
for k in range(labels.max() + 1):                 # parts[0] is the ground, then one part per cluster
    part = parts[k + 1]
    bbox = part.get_axis_aligned_bounding_box()
    print(f"Cluster {k}: {len(part)} points, size: { 'x'.join(f'{x:.2f}' for x in bbox.max_bound - bbox.min_bound) }")
print(f"Noise: {len(parts[-1])} points")

# Matching Two Sets of Points using ICP

An important algorithm for point clouds is Iterative Closest Point (ICP).  It allows us to find the
transformation that maps one point cloud onto another.  The algorithm has the following steps:

1. For each of model and data, subtract their respective mean.  This centres both point clouds at the origin, and
allows estimation of the translation part of the transformation.
2. For each point in one point cloud, find the closest point in the other point cloud, and treat these as correspondences.
3. From the correspondences compute the orientation change using least-square techniques, and use this to update the rotation part of the transformation.
4. Go to step 2 until the correspondence errors fall below a threshold.

We start by subsampling the bunny pointcloud and coloring all those points blue.  This point cloud becomes the
ICP "model".

In [ ]:
model = bunny_pcd.downsample_random(0.1, seed=0)
model.paint("blue")

The ICP "data" that we want to match to the model is another random subsampling of the bunny point
cloud which is then rotated and translated.  The data is colored red.

In [ ]:
from spatialmath import SE3
data = SE3.Trans(0.3, 0.4, 0.5) * SE3.Rz(50, unit="deg") * bunny_pcd.downsample_random(0.05, seed=-1)
data.paint("red")

We can visualize the two point clouds by temporarily concatenating them.  We see the blue model is 
situated close to the origin.

In [ ]:
(model + data).disp(notebook=True)

## Running ICP

`model.ICP(data, ...)` finds the rigid transformation `T` that moves the points of `model` onto the points of `data`.  It also returns a `status` object, which reports how well the two clouds fit together.  The call has these parameters:

* `max_correspondence_distance` is the farthest a point in `data` can be from a point in `model` and still count as a correspondence.  It must be larger than the initial misalignment, which here is about 0.7 m, otherwise there are no correspondences and ICP returns the identity transformation.  A large value is safe for a clean cloud like this one, but with clutter or partial overlap it admits wrong matches, so use the smallest value that works.
* `max_iteration` is the number of times steps 2 and 3 are repeated.
* `relative_fitness` and `relative_rmse` are the convergence tests.  ICP stops early when the fraction of matched points, or the RMS error of the matches, changes by less than these amounts from one iteration to the next.  Setting both to zero switches the tests off, so ICP runs all 2000 iterations.  That takes a little over half a second and gives a more accurate answer than the defaults, which stop after a few iterations.

ICP is a local method: it only improves on the starting alignment, and it can settle into a wrong answer if the clouds start too far apart.  The optional argument `T0` gives it a better initial guess.

In [ ]:
T, status = model.ICP(data, max_correspondence_distance=1, 
                max_iteration=2000, relative_fitness=0, relative_rmse=0)
T.printline()


We see that the estimated transformation, from model to data, matches well with the actual transformation embedded in the code line in an earlier cell.

We can superimpose the two pointclouds by transforming the data by the inverse of the estimated transformation.  Looking closely we can see the distinct red and blue points.

In [ ]:
(model + T.inv() * data).disp(notebook=True)

# Exercises for the reader

* If you are working in a desktop environment, set `desktop=False` in the calls to `disp()`.  This
will launch the proper Open3D browser window which is much more responsive.
  * Learn about the various options supported by this display tool.
* Find some PLY or STL models on the internet and display them.
* Find a depth image online and create a pointcloud using `PointCloud.DepthImage()`.
* If you have access to a [RealSense](https://www.open3d.org/docs/release/tutorial/sensor/realsense.html) or [Kinect](https://www.open3d.org/docs/release/tutorial/sensor/azure_kinect.html) 3D camera, write some Open3D code to read images from it.  You can wrap the Open3D pointcloud objects as MVTB pointclouds using the default constructor `PointCloud(open3d_pointcloud)`
* Add noise to `scene.ply` and see how well the segmentation works
  * how much noise can you add
  * what other parameters does `cluster_dbscan` have?  Experiment with them.
* Experiment with ICP
  * try different transformations for the data
  * add noise to the data point cloud and see how well it works
  * what other parameters does `ICP` have?  Experiment with them.